# Splits on the curated KG

Builds the splits for retraining NetMedGPT on the curated KG (`nodes_preprocessed.csv`, `edges_preprocessed.csv`,
`embeddings_preprocessedKG.pt`: no drug_drug edges, drugs with broad indications and misclassified drug-ADR edges
removed), to test whether performance is preserved.

- **train.csv / valid.csv / test.csv**: the edges of the original split that are also in the curated KG. Edges of the
  drugs removed by the curation are dropped from all three, so training and evaluation use only the curated KG.
- `train.py --inference random_link_split_curated / zero_shot_split_curated` reads these splits together with the
  curated KG files; the negatives are therefore also sampled from the curated KG.
- For the comparison, the original checkpoints are scored on the same curated test pairs (`compare_curated.ipynb`).

Edges are matched by (node type, node ID) of both ends and the relation (`rev_<relation>` rows of the splits are the
same KG relation), independent of direction.
Output: `data/splits_curated/<split folder>/{train,valid,test}.csv` and `summary.csv`.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "curated_kg_training":
    os.chdir("../..")
import sys
sys.path.insert(0, os.getcwd())
import pandas as pd

from netmedgpt.prepare_txgnn_splits import normalize_id

SPLITS = {"random_link_split": "random_{seed}", "zero_shot_split": "complex_disease_{seed}"}
SEEDS = [1, 2, 3, 4, 5]
CURATED_NODES = "data/KG/nodes_preprocessed.csv"
CURATED_EDGES = "data/KG/edges_preprocessed.csv"
OUT_ROOT = "data/splits_curated"

In [ ]:
# ── Curated KG edges as keys: (relation, unordered pair of (type, id)) ───────
nodes_c = pd.read_csv(CURATED_NODES, dtype={"node_id": str}, low_memory=False)
key_of = dict(zip(nodes_c["node_index"], zip(nodes_c["node_type"], nodes_c["node_id"].map(normalize_id))))
curated_drugs = set(nodes_c.loc[nodes_c["node_type"] == "drug", "node_id"])

edges_c = pd.read_csv(CURATED_EDGES, usecols=["relation", "x_index", "y_index"])
curated = set(zip(edges_c["relation"], map(frozenset, zip(edges_c["x_index"].map(key_of), edges_c["y_index"].map(key_of)))))
print(f"curated KG: {len(nodes_c)} nodes, {len(curated)} edges (relation + unordered pair), {len(curated_drugs)} drugs")


def edge_keys(df):
    rel = df["relation"].str.replace(r"^rev_", "", regex=True)
    x = list(zip(df["x_type"], df["x_id"].map(normalize_id)))
    y = list(zip(df["y_type"], df["y_id"].map(normalize_id)))
    return [(r, frozenset((a, b))) for r, a, b in zip(rel, x, y)]

In [ ]:
# ── Restrict train / valid / test to the curated KG ──────────────────────────
summary = []
for inference, folder in SPLITS.items():
    for seed in SEEDS:
        src = os.path.join("data/splits", folder.format(seed=seed))
        dst = os.path.join(OUT_ROOT, folder.format(seed=seed))
        os.makedirs(dst, exist_ok=True)
        row = {"inference": inference, "seed": seed}
        for f in ["train.csv", "valid.csv", "test.csv"]:
            df = pd.read_csv(os.path.join(src, f), low_memory=False)
            keep = pd.Series([k in curated for k in edge_keys(df)], index=df.index)
            df[keep].to_csv(os.path.join(dst, f), index=False)
            name = f.replace(".csv", "")
            row[f"{name}_rows"], row[f"{name}_rows_kept"] = len(df), int(keep.sum())
            if name == "test":
                rel = df["relation"].str.replace(r"^rev_", "", regex=True)
                for r in ["indication", "contraindication", "off-label use", "drug_protein", "drug_effect"]:
                    m = rel == r
                    row[f"test_{r}_kept_pct"] = round(100 * keep[m].mean(), 1) if m.any() else None
        summary.append(row)
        print(f"{inference} seed {seed}: kept train {row['train_rows_kept'] / row['train_rows']:.1%}, "
              f"valid {row['valid_rows_kept'] / row['valid_rows']:.1%}, test {row['test_rows_kept'] / row['test_rows']:.1%} -> {dst}")

summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_ROOT, "summary.csv"), index=False)
summary